Benchmark pipeline for the gng classifier

In [1]:

import torch
import torch.nn as nn
import torch.optim as optim



# -------------------------------------------------
# 1. Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
# -------------------------------------------------
# 2. Load & preprocess data
# -------------------------------------------------
iris = load_iris()
X = iris.data
y = iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [4]:

# Convert to tensors
X_train_gpu = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_train_gpu = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)

In [5]:
epochs = 300
batch_size = 8
lr = 0.001

# -------------------------------------------------
# 3. Define a simple MLP model
# -------------------------------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -------------------------------------------------
# 4. Loss and optimizer
# -------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# -------------------------------------------------
# 5. Training loop
# -------------------------------------------------


dataset = torch.utils.data.TensorDataset(X_train_gpu, y_train_gpu)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(loader):.4f}")

#


Epoch 1/300 - Loss: 1.0757
Epoch 2/300 - Loss: 1.0383
Epoch 3/300 - Loss: 0.9987
Epoch 4/300 - Loss: 0.9504
Epoch 5/300 - Loss: 0.8906
Epoch 6/300 - Loss: 0.8189
Epoch 7/300 - Loss: 0.7322
Epoch 8/300 - Loss: 0.6420
Epoch 9/300 - Loss: 0.5629
Epoch 10/300 - Loss: 0.4939
Epoch 11/300 - Loss: 0.4411
Epoch 12/300 - Loss: 0.4006
Epoch 13/300 - Loss: 0.3714
Epoch 14/300 - Loss: 0.3469
Epoch 15/300 - Loss: 0.3286
Epoch 16/300 - Loss: 0.3140
Epoch 17/300 - Loss: 0.2998
Epoch 18/300 - Loss: 0.2891
Epoch 19/300 - Loss: 0.2788
Epoch 20/300 - Loss: 0.2692
Epoch 21/300 - Loss: 0.2595
Epoch 22/300 - Loss: 0.2502
Epoch 23/300 - Loss: 0.2415
Epoch 24/300 - Loss: 0.2320
Epoch 25/300 - Loss: 0.2228
Epoch 26/300 - Loss: 0.2148
Epoch 27/300 - Loss: 0.2076
Epoch 28/300 - Loss: 0.1969
Epoch 29/300 - Loss: 0.1882
Epoch 30/300 - Loss: 0.1813
Epoch 31/300 - Loss: 0.1718
Epoch 32/300 - Loss: 0.1652
Epoch 33/300 - Loss: 0.1561
Epoch 34/300 - Loss: 0.1494
Epoch 35/300 - Loss: 0.1416
Epoch 36/300 - Loss: 0.1361
E

In [6]:
# 6. Evaluation
# -------------------------------------------------
model.eval()
with torch.no_grad():
    preds = model(X_test_gpu)
    correct = (preds.argmax(dim=1) == y_test_gpu).sum().item()
    acc = correct / len(y_test)

print(f"Test accuracy: {acc:.4f}")

Test accuracy: 0.9667


### Confusion Matrix

In [7]:

y_pred =  preds.argmax(dim=1).cpu().numpy()
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
